# Deep Learning in Audio Applications
### From voice to meeting minutes

You'll turn speech into text, then build the memory that turns text into meaning.
Run cells top to bottom. **YOUR TURN** cells have one line to write. A check tells you if it's right.

<details><summary>More detail: restarting, and what's in each section</summary>

Colab restarted, or you jumped ahead? Run cell 2 again, then every cell above the one you want.

| section | content | minutes |
|---|---|---|
| 1 | Audio to text | ~10 |
| 2 | Where it breaks | ~0 |
| 3 | RNNs: a running memory | ~32 |
| 4 | LSTMs: a memory with a dial | ~50 |
| 5 | Meeting minutes | ~15 |

</details>

## 0. Setup (run once)
On Colab: **Runtime > Change runtime type > T4 GPU**.
Upload `workshop_files.zip` to the Files panel first.

In [ ]:
import os, sys, shutil, subprocess, importlib.util, zipfile
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")                  # fixes a Windows library clash
import warnings; warnings.filterwarnings("ignore", message="FP16")  # Whisper on CPU only

if importlib.util.find_spec("whisper") is None:                       # Whisper: speech-to-text
    print("Installing Whisper. About a minute.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai-whisper"], check=True)
if os.path.exists("workshop_files.zip") and not os.path.exists("prototype"):
    zipfile.ZipFile("workshop_files.zip").extractall(".")              # unpack the workshop files
PROTO = next((p for p in ("prototype", "../prototype") if os.path.isdir(p)), None)   # ../prototype in the repo
PROTO or sys.exit("Upload workshop_files.zip to the Files panel, then run this cell again.")
sys.path.insert(0, PROTO)
print("ffmpeg:", "found" if shutil.which("ffmpeg") else "NOT FOUND -- install it (Colab already has it)")
print("ready")

## 1. Audio to text with Python
Three lines: load Whisper, give it a file, read the text.
Bigger models hear better but run slower. We picked a size that fits your machine.

In [ ]:
import torch, whisper
import live_mic   # not gated behind block 2b -- that cell is optional

MODEL = "small.en" if torch.cuda.is_available() else "base.en"        # a GPU can afford the bigger model
print("Using", MODEL, "on", "the GPU." if torch.cuda.is_available() else "the CPU. About 20s.")
model = whisper.load_model(MODEL)
result = model.transcribe("audio/sample_meeting.wav", language="en")
print(result["text"])

### What Whisper is reading
A spectrogram is sheet music for speech: time left to right, pitch bottom to top.
Here's the waveform and spectrogram for a short clip.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

wave_samples = whisper.load_audio("audio/sample_short.wav")
t = np.arange(len(wave_samples)) / 16000

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax1.plot(t, wave_samples, linewidth=0.5)
ax1.set_ylabel("amplitude")
ax1.set_title("Waveform")
with np.errstate(divide="ignore"):      # a silent stretch would log a divide-by-zero warning
    ax2.specgram(wave_samples, Fs=16000, cmap="magma")
ax2.set_ylabel("pitch (Hz)")
ax2.set_xlabel("time (s)")
ax2.set_title("Spectrogram -- time left to right, pitch bottom to top")
plt.tight_layout()
plt.show()

**Order matters.** Play a clip forwards, then backwards.
Backwards, it stops being words. Same idea for Whisper's spectrogram: reverse it, and the words make no sense.

In [ ]:
from IPython.display import Audio, display

clip = wave_samples[:8000]          # about half a second -- roughly one word
print("Forwards:")
display(Audio(clip, rate=16000))
print("Backwards:")
display(Audio(clip[::-1], rate=16000))

**What the model actually sees.** Not the waveform -- a short window every 10 ms,
turned into a handful of numbers. This is the picture above, in the form a model reads.

In [ ]:
from audio_features import mfcc

samples = whisper.load_audio("audio/sample_short.wav")
frames = mfcc(samples)
print(f"{len(samples) / 16000:.0f}s of audio -> {frames.shape[0]} frames x {frames.shape[1]} numbers")

We also want **who said what**. `get_transcript` starts a new turn
whenever one person pauses (over 1.5s) and the next starts.

In [ ]:
os.environ["WHISPER_MODEL"] = MODEL          # the pipeline's transcriber reads this
from transcriber import get_transcript

print("Transcribing with speaker splitting. About 20s.")
transcript = get_transcript("audio/sample_meeting.wav", pause_gap=1.5)
print(transcript)

## 2. Where transcription struggles
Real recordings aren't clean. Three things hurt most: **noise**, **a poor mic**, **accents**.
We score by `word_accuracy`: 1.0 means every word.

In [ ]:
import numpy as np
from audio_check import word_accuracy

SAID = ("Venue is confirmed for the 14th, catering still pending. "
        "Can you follow up with catering by Friday? Yes, I'll email them today.")
clean = whisper.load_audio("audio/sample_short.wav")          # the same three sentences, 14s

def hear(samples):
    return model.transcribe(samples, language="en")["text"].strip()

print("Clean:", f"{word_accuracy(SAID, hear(clean)):.0%}", "|", hear(clean), "(about 5s)")

**Noise.** Add random hiss. How loud before Whisper breaks?

In [ ]:
NOISE = 0.1
noisy = (clean + NOISE * np.random.default_rng(0).normal(size=clean.shape)).astype("float32")
heard = hear(noisy)
print(f"Noise {NOISE}:", f"{word_accuracy(SAID, heard):.0%}", "|", heard)

**A poor mic.** Keep only every 4th sample, like a cheap mic or a phone line.
The high sounds (s, t, f) disappear.

In [ ]:
poor = np.repeat(clean[::4], 4).astype("float32")
heard = hear(poor)
print("Poor mic:", f"{word_accuracy(SAID, heard):.0%}", "|", heard)

**Accents.** Record yourself reading the same three sentences on your phone:

> Venue is confirmed for the 14th, catering still pending. / Can you follow up with catering by Friday? / Yes, I'll email them today.

Upload it, put the filename below.

In [ ]:
MY_VOICE = None            # for example "my_voice.m4a"
if MY_VOICE:
    mine = whisper.load_audio(MY_VOICE)
    heard = hear(mine)
    print("My voice:", f"{word_accuracy(SAID, heard):.0%}", "|", heard)
else:
    print("Set MY_VOICE to a file you uploaded, then run this cell again.")

**Kahoot: which is most accurate?** Four clips, same sentence: ideal, then three noisy
versions. Vote in Kahoot on the ranking before running the answer key below.

In [ ]:
NOISY_CLIPS = [
    ("1. ideal", "noisy_1_ideal.wav"),
    ("2. background hiss", "noisy_2_hiss.wav"),
    ("3. poor microphone", "noisy_3_poor_mic.wav"),
    ("4. a second voice underneath", "noisy_4_second_voice.wav"),
]

print("Vote now: which order, best to worst?")
print()
for label, filename in NOISY_CLIPS:
    print(label)
    display(Audio(f"audio/{filename}"))

**Answer key.** Measured live, not assumed.

<details><summary>More detail: the numbers, by Whisper model size</summary>

Percentages depend on model size (`MODEL`), not the clips. CPU runs:

| model | ideal | second voice | hiss | poor mic |
|---|---|---|---|---|
| `base.en` | 100% | 61% | 52% | 35% |
| `small.en` (this notebook's GPU default) | 100% | 87% | 83% | 22% |
| `medium.en` | 100% | 91% | 83% | 35% |

The ranking held across all three sizes. Run the cell below to get yours.

</details>

In [ ]:
print("Running Whisper on all four clips. About 20s.")
for label, filename in NOISY_CLIPS:
    heard = hear(whisper.load_audio(f"audio/{filename}"))
    print(f"{label:32s} {word_accuracy(SAID, heard):.0%}  | {heard}")

### 2b. Say it yourself (live mic) *(optional)*
**Predict:** real background noise gets mixed in. How much will accuracy drop?

In [ ]:
import live_mic

SAID_LIVE, clear_live_path, used_fallback = live_mic.say("block2_clear", model, seconds=5)
print("Heard:", SAID_LIVE)

clear_live = whisper.load_audio(clear_live_path)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
t = np.arange(len(clear_live)) / 16000
ax1.plot(t, clear_live, linewidth=0.5)
ax1.set_ylabel("amplitude")
ax1.set_title("Your waveform")
with np.errstate(divide="ignore"):
    ax2.specgram(clear_live, Fs=16000, cmap="magma")
ax2.set_ylabel("pitch (Hz)")
ax2.set_xlabel("time (s)")
ax2.set_title("Your spectrogram")
plt.tight_layout()
plt.show()

**Try it:** 1. Run the cell. 2. Say the same sentence again, with real noise --
tap the desk, rustle paper, play music.

In [ ]:
from audio_check import word_accuracy

NOISY_LIVE, noisy_live_path, _ = live_mic.say("block2_noisy", model, seconds=5)
print("Heard:", NOISY_LIVE)
noisy_accuracy = word_accuracy(SAID_LIVE, NOISY_LIVE)
print(f"Accuracy vs your clear recording: {noisy_accuracy:.0%}")
if noisy_accuracy >= 1.0:
    print("Whisper coped. Try louder noise to see it drop.")

**Why:** your exact sound is what Whisper reads. Add noise and accuracy often drops --
though a short, clear sentence can survive a lot of it.

**Try it:** record one word. Hear it played backwards.

In [ ]:
WORD_LIVE, word_live_path, _ = live_mic.say("block2_backward", model, seconds=3, minimum_words=1)
print("Heard:", WORD_LIVE)

word_samples = whisper.load_audio(word_live_path)
print("Forwards:")
display(Audio(word_samples, rate=16000))
print("Backwards:")
display(Audio(word_samples[::-1].copy(), rate=16000))
backward_heard = model.transcribe(word_samples[::-1].copy(), language="en")["text"].strip()
print("Whisper on the reversed clip:", repr(backward_heard) if backward_heard else "(nothing)")

**Why:** order matters to Whisper, same as the RNN two sections below.
Reversed, your word stops being a word it recognises.

## 3. RNNs: reading words as a sequence
It reads one **word** at a time, carrying a running memory forward:
`h = tanh(W_x.x + W_h.h_prev + b)`

<details><summary>More detail</summary>

Whisper reads its spectrogram in time order too -- same "order matters" idea as the reversed word.

</details>

In [ ]:
import numpy as np
from lstm_model import model_words

sentence = "Can you follow up with catering by Friday?"
words = model_words(sentence)
vocab = {w: i for i, w in enumerate(sorted(set(words)))}
numbers = [vocab[w] for w in words]

print("Words  :", words)
print("Numbers:", numbers, "-- each word gets its own number")

# Toy RNN, one word at a time -- random weights, not the trained model (that's in section 4).
rng = np.random.default_rng(0)
dim = 4
toy_embeddings = rng.normal(size=(len(vocab), dim))
W_x_toy, W_h_toy, b_toy = rng.normal(scale=0.5, size=(dim, dim)), rng.normal(scale=0.5, size=(dim, dim)), np.zeros(dim)

print()
print("Stepping the RNN one word at a time:")
memory = np.zeros(dim)
for word, number in zip(words, numbers):
    x = toy_embeddings[number]
    memory = np.tanh(W_x_toy @ x + W_h_toy @ memory + b_toy)
    print(f"  after {word!r:10s} memory = {np.round(memory, 2)}")

### 3b. Watch it read YOUR sentence (live mic)
**Predict:** does the memory update at every word, or only some?

1. Run the cell.
2. Say any sentence, 5-15 words.

In [ ]:
import toy_memory

print(f"Toy memory keeps {toy_memory.KEEP_RATE:.0%} of the old memory each word (toy setting, not learned).")

SENTENCE_LIVE, sentence_live_path, _ = live_mic.say("block3_sentence", model, seconds=5)
print("Heard:", SENTENCE_LIVE)
live_words = model_words(SENTENCE_LIVE)

h = np.zeros(8)
mem_changes = []
for w in live_words:
    h_new = toy_memory.step(live_mic.word_vector(w), h)
    mem_changes.append(float(np.abs(h_new - h).mean()))
    h = h_new

plt.figure(figsize=(8, 3))
plt.bar(range(len(live_words)), mem_changes)
plt.xticks(range(len(live_words)), live_words, rotation=45, ha="right")
plt.ylabel("change in memory")
plt.title("How much the memory changes at each of your words")
plt.tight_layout()
plt.show()
print(f"Changed at every one of your {len(live_words)} words. No two updates were the same size.")

**Why:** this plots the CHANGE each word makes, not the memory's size.
Size can look similar word to word even while the memory keeps moving.
`toy_memory` is a stand-in for a real RNN step -- a fixed formula, not learned.

### YOUR TURN (Part A): one step of an RNN
**Predict:** once this returns a real memory, will the plot below still look the same?

Write one line: `new memory = tanh(W_x @ x + W_h @ h_prev + b)`.
`@` is matrix multiply, `tanh` is `np.tanh`.

*`x`: this word. `h_prev`: memory so far. `W_x`, `W_h`: learned weights. `b`: a small nudge.*

In [ ]:
import numpy as np, hands_on

def rnn_step(x, h_prev, W_x, W_h, b):
    # TODO: return the new memory (the line above, in numpy)
    return None

hands_on.rnn_step = rnn_step
hands_on.MEMORY_ON = True
hands_on.part_a()

<details><summary>Stuck? (click to expand)</summary>

- **A grid, not a list of numbers** -- you used `*` where `@` should be.
- **A number outside -1..1** -- `tanh` is missing.
- **Close, not quite** -- check you added `+ b`.
- **Still `None`** -- replace `return None` with your formula.

</details>

Now **break the memory**. Delete `+ W_h @ h_prev` from your line (not a flag --
the term itself). Set `hands_on.MEMORY_ON = False`. Run again.

Ignore the line printed above -- it's only half the fix. The cell below shows the same break.

In [ ]:
# Amnesia version: the old-memory term is gone from the line, not just flagged off.
def amnesia_rnn_step(x, h_prev, W_x, W_h, b):
    return np.tanh(W_x @ x + b)     # no + W_h @ h_prev

hands_on.rnn_step = amnesia_rnn_step
hands_on.MEMORY_ON = False
hands_on.part_a()

### 4b. Prove it with your own voice (live mic) *(bonus)*
**Predict:** will "memory on" tell two sentences apart by the last word, if only the first word differs? Will "memory off"?

Say these two lines, one per cell (close is fine, doesn't need to match word for word):
- "Cats are sitting near the window this evening."
- "Dogs are sitting near the window this evening." 

In [ ]:
CATS_LIVE, _, _ = live_mic.say("block4_cats", model, seconds=5)
print("Heard (cats):", CATS_LIVE)

In [ ]:
DOGS_LIVE, _, _ = live_mic.say("block4_dogs", model, seconds=5)
print("Heard (dogs):", DOGS_LIVE)

cats_words = model_words(CATS_LIVE)
dogs_words_spoken = model_words(DOGS_LIVE)
dogs_words = toy_memory.first_word_pair(cats_words, dogs_words_spoken)
print(f"Comparing against: {' '.join(dogs_words)!r} -- your second recording's first word "
      f"('{dogs_words[0]}') plus the first recording's later words.")

if dogs_words and cats_words and dogs_words[0] == cats_words[0]:
    print(f"Both start with {cats_words[0]!r}. Using a placeholder word so the plot still works.")
    dogs_words = ["different"] + dogs_words[1:]

n_live = len(cats_words)
gaps_on = toy_memory.gap_curve(cats_words, dogs_words, live_mic.word_vector, keep_rate=toy_memory.KEEP_RATE)
gaps_off = toy_memory.gap_curve(cats_words, dogs_words, live_mic.word_vector, keep_rate=0.0)
pct_on = [g / gaps_on[0] * 100 for g in gaps_on]
pct_off = [g / gaps_off[0] * 100 for g in gaps_off]

plt.figure(figsize=(8, 3))
plt.plot(range(1, n_live + 1), pct_on, marker="o", label=f"memory ON (keeps {toy_memory.KEEP_RATE:.0%}/word)")
plt.plot(range(1, n_live + 1), pct_off, marker="o", label="memory OFF (amnesia)")
plt.xlabel("word")
plt.ylabel("% of word 1's difference remaining")
plt.title("Does it still remember your first word was different?")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Memory ON:  {pct_on[-1]:.1f}% left by word {n_live}. Keeps {toy_memory.KEEP_RATE:.0%} every word. Never zero.")
print(f"Memory OFF: {pct_off[-1]:.1f}% left by word {n_live}. Exactly 0% from word 2 on, every time.")

**Why:** the keep rate is stated (0.8), so the decay is exact, not just usual.
Memory OFF has no history term, so it's exactly 0% from word 2, every time.
The gap you see is guaranteed by the formula, on words you actually said.

### YOUR TURN: write your own first-word-swap pair
Write two sentences that differ in ONLY the first word, like Cats/Dogs above.
The check confirms it.

In [ ]:
MY_SENTENCE_A = "Cats are sitting by the door tonight."   # YOUR TURN: write your own pair here
MY_SENTENCE_B = "Dogs are sitting by the door tonight."

my_words_a = model_words(MY_SENTENCE_A)
my_words_b = model_words(MY_SENTENCE_B)

if len(my_words_a) != len(my_words_b):
    print(f"Not yet. {len(my_words_a)} words vs {len(my_words_b)}. Match the word counts first.")
elif my_words_a and my_words_a[0] == my_words_b[0]:
    print("Not yet. The first word needs to actually be different.")
elif my_words_a[1:] != my_words_b[1:]:
    diff_at = next(i for i in range(1, len(my_words_a)) if my_words_a[i] != my_words_b[i])
    print(f"Not yet. Word {diff_at + 1} also differs ({my_words_a[diff_at]!r} vs {my_words_b[diff_at]!r}). Only the first word should.")
else:
    print("Check: OK. Only the first word differs.")
    my_gaps = toy_memory.gap_curve(my_words_a, my_words_b, live_mic.word_vector)
    print(f"Your pair: {my_gaps[-1] / my_gaps[0] * 100:.1f}% of the first word's effect left at the end.")

<details><summary>Stuck? (click to expand)</summary>

- Different word counts -- add or remove a word until they match.
- A later word differs too -- copy sentence A, change only the first word.
- Skipping this is fine. Nothing below depends on it.

</details>

### Why simple memory fades
**Predict:** after about 47 steps, is there still a useful amount of the signal left?

Like broken telephone: pass a message through enough people and it's gone.
An RNN's memory fades the same way, every step. This is an illustration, not the trained model.

This is why **long-range dependencies** matter: an early line a later line depends on can get lost.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Part A's toy RNN, replayed with the correct line -- same setup, seed 1.
toy_rng = np.random.default_rng(1)
toy_hidden = 8
toy_W_x, toy_W_h, toy_b = (toy_rng.normal(0, 0.5, (toy_hidden, 4)), toy_rng.normal(0, 0.25, (toy_hidden, toy_hidden)), np.zeros(toy_hidden))
toy_words_a = toy_rng.normal(size=(12, 4))
toy_words_b = toy_words_a.copy()
toy_words_b[0] = toy_rng.normal(size=4)
toy_h_a = toy_h_b = np.zeros(toy_hidden)
toy_gaps = []
for x_a, x_b in zip(toy_words_a, toy_words_b):
    toy_h_a = np.tanh(toy_W_x @ x_a + toy_W_h @ toy_h_a + toy_b)
    toy_h_b = np.tanh(toy_W_x @ x_b + toy_W_h @ toy_h_b + toy_b)
    toy_gaps.append(np.abs(toy_h_a - toy_h_b).sum())
toy_kept = np.mean([toy_gaps[k + 1] / toy_gaps[k] for k in range(len(toy_gaps) - 1) if toy_gaps[k] > 1e-12])

plt.figure(figsize=(7, 4))
n = np.arange(48)
for r, note in [(0.49, "untrained setting"), (0.95, "keeping 95%")]:
    retained = r ** n
    plt.plot(n, retained * 100, label=f"r = {r} ({note}): {retained[-1]:.1%} left at n=47")
plt.plot(n, (toy_kept ** n) * 100, linestyle=":", color="gray",
         label=f"Part A's toy RNN (r = {toy_kept:.2f}, measured here): same fading")
plt.xlabel("steps (n)")
plt.ylabel("% of the original signal retained")
plt.title("Memory retention: r^n (an illustration, not measured)", fontsize=11)
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8, borderaxespad=0)
plt.ylim(0, 100)
plt.show()

print(f"0.49^47 = {0.49**47:.4%}   0.95^47 = {0.95**47:.1%}   toy RNN: r={toy_kept:.3f}, {toy_kept**47:.4%} left at n=47")

### 5b. Does a SHORT swap survive further than a LONG one? (live mic)
**Predict:** a bigger trace of the first word after 4 words, or after 15+?

Say:
- "Red cars drive fast." / "Blue cars drive fast." (4 words)

*(bonus)* Then the longer pair:
- "Monday mornings always feel quieter than every other weekday afternoon session we have had this term." / the same starting with "Tuesday" (16 words)

In [ ]:
SHORT_A, _, _ = live_mic.say("block5_short_a", model, seconds=4)
print("Heard (short a):", SHORT_A)

In [ ]:
SHORT_B, _, _ = live_mic.say("block5_short_b", model, seconds=4)
print("Heard (short b):", SHORT_B)

In [ ]:
LONG_A, _, _ = live_mic.say("block5_long_a", model, seconds=8)
print("Heard (long a):", LONG_A)

In [ ]:
LONG_B, _, _ = live_mic.say("block5_long_b", model, seconds=8)
print("Heard (long b):", LONG_B)

short_words_a = model_words(SHORT_A)
short_words_b = toy_memory.first_word_pair(short_words_a, model_words(SHORT_B))
long_words_a = model_words(LONG_A)
long_words_b = toy_memory.first_word_pair(long_words_a, model_words(LONG_B))

if short_words_a and short_words_b and short_words_a[0] == short_words_b[0]:
    print(f"Short pair: both start with {short_words_a[0]!r}. Using a placeholder word.")
    short_words_b = ["different"] + short_words_b[1:]
if long_words_a and long_words_b and long_words_a[0] == long_words_b[0]:
    print(f"Long pair: both start with {long_words_a[0]!r}. Using a placeholder word.")
    long_words_b = ["different"] + long_words_b[1:]

short_gaps = toy_memory.gap_curve(short_words_a, short_words_b, live_mic.word_vector)
long_gaps = toy_memory.gap_curve(long_words_a, long_words_b, live_mic.word_vector)
short_pct = short_gaps[-1] / short_gaps[0] * 100
long_pct = long_gaps[-1] / long_gaps[0] * 100
short_below10 = toy_memory.words_below(short_gaps, 10)
long_below10 = toy_memory.words_below(long_gaps, 10)

print(f"Toy memory keeps {toy_memory.KEEP_RATE:.0%} each word (toy setting, not learned).")

plt.figure(figsize=(5, 3.5))
bars = plt.bar([f"short pair{chr(10)}({len(short_words_a)} words)", f"long pair{chr(10)}({len(long_words_a)} words)"],
               [short_pct, long_pct], color=["green", "red"])
plt.ylabel("% of the first word's difference left at the end")
plt.title("How much of your first word survives?")
plt.ylim(0, 100)
for bar, v in zip(bars, [short_pct, long_pct]):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 2, f"{v:.1f}%", ha="center")
plt.tight_layout()
plt.show()

short_note = f", falls below 10% after word {short_below10}" if short_below10 else " -- never falls below 10%"
long_note = f", falls below 10% after word {long_below10}" if long_below10 else " -- never falls below 10%"
print(f"Short pair ({len(short_words_a)} words): {short_pct:.1f}% of the first word's effect left{short_note}")
print(f"Long pair  ({len(long_words_a)} words): {long_pct:.2f}% of the first word's effect left{long_note}")

**Why:** keep rate 0.8 means the first word's effect decays as exactly `0.8^(word-1)`.
A 4-word pair keeps about half; a 16-word pair is down near 3%. Checked over 500 random pairs: no variance.
Not the trained model -- that stacks 5 LSTM layers, not one step.

### 5c. The headline: short exchange vs long conversation (live mic)
**Predict:** will a short exchange "remember" an early fact? Will a long one?

Say a SHORT exchange, 2-3 turns, ~2s pause, one short reply each, first turn holds one fact
(e.g. "The meeting is on Friday at three."):

In [ ]:
from transcriber import segments_to_transcript

SHORT_CONV, short_conv_path, _ = live_mic.say("headline_short_conv", model, seconds=15, minimum_words=6)
print("Heard (flat):", SHORT_CONV)

short_result = model.transcribe(short_conv_path, word_timestamps=True, language="en")
short_words_ts = [{"start": w["start"], "end": w["end"], "text": w["word"]}
                  for seg in short_result["segments"] for w in seg.get("words", [])]
short_turn_text = segments_to_transcript(short_words_ts, pause_gap=1.2)
short_turn_lines = [l for l in short_turn_text.strip().split(chr(10)) if l]
short_conv_turns = [(f"Turn{i + 1}", line.split(": ", 1)[1]) for i, line in enumerate(short_turn_lines)]
print("Turns heard:", short_conv_turns)

Here's a long one we recorded earlier, so you see the result right away.
*(bonus)* Set `RECORD_OWN_LONG = True` below to record your own: 8+ turns, ~60s, same first-line fact.

In [ ]:
from lstm_model import tokenize_transcript

RECORD_OWN_LONG = False   # (bonus) set True to record your own long conversation instead

if RECORD_OWN_LONG:
    LONG_CONV, long_conv_path, _ = live_mic.say("headline_long_conv", model, seconds=65, minimum_words=20)
    print("Heard (flat):", LONG_CONV)
else:
    print("Here's a long one we recorded earlier.")
    long_conv_path = os.path.join(live_mic.LIVE_AUDIO_DIR, "headline_long_conv.wav")
    LONG_CONV = live_mic.hear(long_conv_path, model)
    print("Heard (flat):", LONG_CONV)

long_result = model.transcribe(long_conv_path, word_timestamps=True, language="en")
long_words_ts = [{"start": w["start"], "end": w["end"], "text": w["word"]}
                 for seg in long_result["segments"] for w in seg.get("words", [])]
long_turn_text = segments_to_transcript(long_words_ts, pause_gap=1.2)
long_turn_lines = [l for l in long_turn_text.strip().split(chr(10)) if l]
long_conv_turns = [(f"Turn{i + 1}", line.split(": ", 1)[1]) for i, line in enumerate(long_turn_lines)]
print("Turns heard:", long_conv_turns)

short_tokens, _ = tokenize_transcript(short_conv_turns)
long_tokens, _ = tokenize_transcript(long_conv_turns)
ALT_FIRST_LINE = model_words("The meeting is on Monday at nine.")

short_tokens_b = toy_memory.first_line_pair(short_tokens, ALT_FIRST_LINE)
long_tokens_b = toy_memory.first_line_pair(long_tokens, ALT_FIRST_LINE)

short_gaps = toy_memory.gap_curve(short_tokens, short_tokens_b, live_mic.word_vector, keep_rate=toy_memory.HEADLINE_KEEP_RATE)
long_gaps = toy_memory.gap_curve(long_tokens, long_tokens_b, live_mic.word_vector, keep_rate=toy_memory.HEADLINE_KEEP_RATE)
# Measured from the END of line 1, not word 1 -- a line swap can share its
# first word or two with the alt line, so the word-1 gap can read 0 even
# though the lines differ.
short_ref = short_gaps[toy_memory.line_boundary_index(short_tokens)]
long_ref = long_gaps[toy_memory.line_boundary_index(long_tokens)]
short_pct_conv = short_gaps[-1] / short_ref * 100
long_pct_conv = long_gaps[-1] / long_ref * 100

print(f"Toy setting: keeps {toy_memory.HEADLINE_KEEP_RATE:.0%} per word.")
plt.figure(figsize=(5, 3.5))
bars = plt.bar(["short exchange", "long conversation"], [short_pct_conv, long_pct_conv], color=["green", "red"])
plt.ylabel("% of line 1's fact still in memory at the end")
plt.title("Does the toy RNN remember line 1 by the end?")
plt.ylim(0, 100)
for bar, v in zip(bars, [short_pct_conv, long_pct_conv]):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 2, f"{v:.1f}%", ha="center")
plt.tight_layout()
plt.show()
print(f"Short exchange ({len(short_tokens)} tokens): {short_pct_conv:.1f}% of line 1's fact left.")
print(f"Long conversation ({len(long_tokens)} tokens): {long_pct_conv:.2f}% of line 1's fact left.")

**Why:** keep one short reply each, so the short bar stays clearly above zero.
A short exchange keeps a real share of line 1. A long one runs it down to almost nothing.
Block 8 revisits this with the LSTM dial. Block 9 checks the trained model.

## 4. LSTMs: an eraser you can set
An LSTM adds a **dial**. Its memory is a **notebook** with three switches:
**eraser** (forget), **pencil** (input), **voice** (output).
`c = f*c_prev + i*g`

<details><summary>More detail: the hourglass shape, and the end-of-turn token</summary>

Training sets the dial; Hands-on B lets you turn it. Our model stacks 5 layers, like an
hourglass: the middle layer looks at four words at a time. A `<eot>` token between turns
is the only signal someone new is talking.

</details>

In [ ]:
from lstm_model import run_pipeline, load_trained_model, tokenize_transcript, pad_to_multiple_of_4

turns, output, turn_ends = run_pipeline(transcript=transcript)
trained = load_trained_model()
vocab = {w: i for i, w in enumerate(trained["vocab"])}
tokens, _ends = tokenize_transcript(turns)
x = trained["embeddings"][[vocab.get(w, 0) for w in tokens]]
_hourglass_out, hourglass_layers = trained["model"].forward(pad_to_multiple_of_4(x))

print("The hourglass: 5 layers, this meeting:")
print(f"  {'layer':<9}{'units':>7}{'time steps':>13}")
for name, out in hourglass_layers.items():
    steps, units = out.shape
    print(f"  {name:<9}{units:>7}{steps:>13}")

layer_names = list(hourglass_layers.keys())
layer_units = [out.shape[1] for out in hourglass_layers.values()]
plt.figure(figsize=(6, 2.5))
plt.bar(layer_names, layer_units)
plt.ylabel("units")
plt.title("The hourglass shape: units per layer")
plt.show()

**Watch the gates.** The eraser, pencil and voice are real numbers the model computes.
Layer 1, replayed by hand over "Can you book the room? / Sure.":

*(One layer, one ensemble member -- a sketch, not the whole model.)*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lstm_model import TURN_BOUNDARY_TOKEN, tokenize_transcript

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

gate_example = [("Mei", "Can you book the room?"), ("Ravi", "Sure.")]
gate_tokens, _gate_ends = tokenize_transcript(gate_example)
gate_ids = [vocab.get(w, 0) for w in gate_tokens]
gate_x_seq = trained["embeddings"][gate_ids]
layer1 = trained["model"].layer1

h, c = np.zeros(layer1.hidden_size), np.zeros(layer1.hidden_size)
erasers, pencils, voices = [], [], []
for x in gate_x_seq:
    combined = np.concatenate([h, x])
    f = sigmoid(layer1.W_f @ combined + layer1.b_f)
    i_gate = sigmoid(layer1.W_i @ combined + layer1.b_i)
    c_tilde = np.tanh(layer1.W_c @ combined + layer1.b_c)
    c = f * c + i_gate * c_tilde
    o = sigmoid(layer1.W_o @ combined + layer1.b_o)
    h = o * np.tanh(c)
    erasers.append(f.mean())
    pencils.append(i_gate.mean())
    voices.append(o.mean())

x_pos = np.arange(len(gate_tokens))
width = 0.25
plt.figure(figsize=(9, 3))
plt.bar(x_pos - width, erasers, width, label="eraser (forget)")
plt.bar(x_pos, pencils, width, label="pencil (input)")
plt.bar(x_pos + width, voices, width, label="voice (output)")
plt.xticks(x_pos, gate_tokens, rotation=45, ha="right")
plt.ylabel("gate value (mean of 8 units)")
plt.title("Layer 1's gates, one word at a time (member 0)")
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8, borderaxespad=0)
plt.tight_layout()
plt.show()

for tok, f_m, i_m, o_m in zip(gate_tokens, erasers, pencils, voices):
    marker = "  <- speaker changed" if tok == TURN_BOUNDARY_TOKEN else ""
    print(f"{tok:10s} eraser={f_m:.2f}  pencil={i_m:.2f}  voice={o_m:.2f}{marker}")

`<eot>` is the model's only clue a new speaker is talking -- no name is in the input.
Here it is in a 2-turn exchange:

In [ ]:
from lstm_model import TURN_BOUNDARY_TOKEN, tokenize_transcript

example = [("Alex", "Can you follow up with catering by Friday?"), ("Priya", "Yes, I'll email them today.")]
example_tokens, example_ends = tokenize_transcript(example)
print(example_tokens)
print(f"{TURN_BOUNDARY_TOKEN!r} marks the speaker change (turn_end_indices = {example_ends}).")

In [ ]:
import numpy as np
from lstm_model import fold_time, unfold_time
x = np.arange(16.0).reshape(8, 2)
print("8 steps of 2 numbers, folded:", fold_time(x).shape, "(4 steps of 4). Unfolded matches:", np.array_equal(unfold_time(fold_time(x)), x))

### 6b. Watch the gates on YOUR turns (live mic)
**Predict:** where does the eraser dip most -- mid-sentence, or at the speaker change?

1. Run the cell.
2. Say two short turns, ~2s pause -- a question and a one-word answer.

In [ ]:
from transcriber import segments_to_transcript
from lstm_model import tokenize_transcript, TURN_BOUNDARY_TOKEN

GATE_LIVE, gate_path, _ = live_mic.say("block6_gates", model, seconds=8)
print("Heard (flat):", GATE_LIVE)

gate_result = model.transcribe(gate_path, word_timestamps=True, language="en")
gate_words_ts = [{"start": w["start"], "end": w["end"], "text": w["word"]}
                  for seg in gate_result["segments"] for w in seg.get("words", [])]
gate_turn_text = segments_to_transcript(gate_words_ts, pause_gap=1.2)
gate_turn_lines = [l for l in gate_turn_text.strip().split(chr(10)) if l]
gate_live_turns = [(f"Turn{i + 1}", line.split(": ", 1)[1]) for i, line in enumerate(gate_turn_lines)]
print("Turns heard:", gate_live_turns)

if len(gate_live_turns) < 2:
    print("Heard it as one turn. Leave a 2s pause, then run this cell again.")

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

gate_live_tokens, _gate_live_ends = tokenize_transcript(gate_live_turns)
gate_live_ids = [vocab.get(w, 0) for w in gate_live_tokens]
gate_live_x_seq = trained["embeddings"][gate_live_ids]
live_layer1 = trained["model"].layer1   # the trained model's own layer 1 (member 0), not a toy

h, c = np.zeros(live_layer1.hidden_size), np.zeros(live_layer1.hidden_size)
live_erasers, live_pencils = [], []
for x in gate_live_x_seq:
    combined = np.concatenate([h, x])
    f = sigmoid(live_layer1.W_f @ combined + live_layer1.b_f)
    i_gate = sigmoid(live_layer1.W_i @ combined + live_layer1.b_i)
    c_tilde = np.tanh(live_layer1.W_c @ combined + live_layer1.b_c)
    c = f * c + i_gate * c_tilde
    h = sigmoid(live_layer1.W_o @ combined + live_layer1.b_o) * np.tanh(c)
    live_erasers.append(f.mean())
    live_pencils.append(i_gate.mean())

x_pos = np.arange(len(gate_live_tokens))
width = 0.35
plt.figure(figsize=(9, 3))
plt.bar(x_pos - width / 2, live_erasers, width, label="eraser (forget)")
plt.bar(x_pos + width / 2, live_pencils, width, label="pencil (input)")
plt.xticks(x_pos, gate_live_tokens, rotation=45, ha="right")
plt.ylabel("gate value (mean of 8 units)")
plt.title("Layer 1's gates over YOUR two turns (trained model, member 0)")
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8, borderaxespad=0)
plt.tight_layout()
plt.show()

eot_positions = [i for i, t in enumerate(gate_live_tokens) if t == TURN_BOUNDARY_TOKEN]
if eot_positions:
    eot_eraser = live_erasers[eot_positions[0]]
    other = [e for i, e in enumerate(live_erasers) if i not in eot_positions]
    print(f"Eraser at <eot> = {eot_eraser:.2f}. Everywhere else: {min(other):.2f}-{max(other):.2f}.")
else:
    print("No <eot> heard. Say two clearly separate turns with a ~2s pause.")

**Why:** the trained model's own layer 1, one member -- a sketch, not the whole model.
*(Checked on 1000 exchanges: eraser is lowest at `<eot>` about 82% of the time -- usually, not always.)*

Now try your Cats/Dogs pair from 4b through an untrained toy LSTM, at three knob settings.

In [ ]:
from lstm_model import SimpleLSTM

# Reuses cats_words/dogs_words from 4b if it ran. If 4b was skipped, get the
# pair now instead, so this cell never depends on a skipped bonus cell.
if "cats_words" not in globals() or "dogs_words" not in globals():
    print("4b wasn't run. Getting the Cats/Dogs pair now instead.")
    _cats_live, _, _ = live_mic.say("block4_cats", model, seconds=5)
    _dogs_live, _, _ = live_mic.say("block4_dogs", model, seconds=5)
    cats_words = model_words(_cats_live)
    dogs_words = toy_memory.first_word_pair(cats_words, model_words(_dogs_live))
    if cats_words and dogs_words and cats_words[0] == dogs_words[0]:
        print(f"Both start with {cats_words[0]!r}. Using a placeholder word.")
        dogs_words = ["different"] + dogs_words[1:]

n_l = len(cats_words)

def lstm_notebook_gap(words_a, words_b, knob):
    layer = SimpleLSTM(8, 8, seed=5)
    layer.b_f = layer.b_f + knob
    ha = hb = ca = cb = np.zeros(8)
    gaps = []
    for wa, wb in zip(words_a, words_b):
        xa, xb = live_mic.word_vector(wa), live_mic.word_vector(wb)
        ha, ca = layer.step(xa, ha, ca)
        hb, cb = layer.step(xb, hb, cb)
        gaps.append(np.abs(ca - cb).sum())
    return gaps

knob_gaps = {knob: lstm_notebook_gap(cats_words[:n_l], dogs_words[:n_l], knob) for knob in (-6, 0, 6)}

plt.figure(figsize=(8, 3))
for knob, gaps in knob_gaps.items():
    plt.plot(range(1, n_l + 1), gaps, marker="o", label=f"knob {knob:+d}")
plt.xlabel("word")
plt.ylabel("|notebook(cats) - notebook(dogs)|")
plt.title("Your first-word swap through the toy LSTM, three knob settings")
plt.legend()
plt.tight_layout()
plt.show()

for knob, gaps in knob_gaps.items():
    note = "erases fastest" if knob == -6 else ("keeps longest" if knob == 6 else "default")
    print(f"Knob {knob:+d} ({note}): gap at word 2 = {gaps[1]:.2f}, by word {n_l} = {gaps[-1]:.4f}")

**Why:** the eraser knob decides how long your first word survives.
More negative erases faster. More positive keeps it longer. On your own words.

### Teaching it, and catching it out
`label -> prediction -> loss -> update`.
`loss = -log(p(correct label))`: right and confident is a tiny loss; wrong and confident is large.

<details><summary>More detail: the analogy</summary>

Like marking practice papers: the model guesses, is told how wrong, adjusts a little.
Worked example, this notebook's own meeting:

</details>

In [ ]:
from lstm_model import label_probabilities
from meeting_minutes import label_turns_stub

gold = label_turns_stub(turns)
probs = label_probabilities(turns, trained)

for (speaker, text), g, p in zip(turns, gold, probs):
    if "email them today" in text or "Venue is confirmed" in text:
        correct = g["category"]
        p_correct = p[trained["labels"].index(correct)]
        print(text)
        print(f"  gold = {correct}, p(correct) = {p_correct:.4f}, loss = {-np.log(p_correct):.2f}")
        print()

Confident and right on "I'll email them today." Confident and **wrong** on
"Venue is confirmed..." -- a decision stated as fact. That's the biggest loss here.

<details><summary>More detail: keyword rules vs. the trained model, across all held-out meetings</summary>

A keyword checklist vs. the trained model, on a held-out meeting never seen in training.
This is the model's best-case meeting, not typical: across 17 held-out meetings, the
trained LSTM scores 131/149 (0.879) and 104/122 (0.852) against the keyword checklist's
111/149 (0.745) and 93/122 (0.762) -- a real improvement, but not perfect. On 3 of the 17
meetings the keyword checklist actually wins.

</details>

In [ ]:
from pipeline import load_text
from meeting_minutes import label_turns_keywords, label_turns_model

held_out_text, held_out_gold = load_text(f"{PROTO}/demo_student_council.txt")
held_out_turns, held_out_output, held_out_ends = run_pipeline(transcript=held_out_text)
kw_labels = label_turns_keywords(held_out_turns)
lstm_labels = label_turns_model(held_out_turns, held_out_output, held_out_ends)

kw_right = sum(k["category"] == g for k, g in zip(kw_labels, held_out_gold))
lstm_right = sum(l["category"] == g for l, g in zip(lstm_labels, held_out_gold))
print(f"Keyword rules: {kw_right}/{len(held_out_turns)} turns right")
print(f"Trained LSTM:  {lstm_right}/{len(held_out_turns)} turns right")
print()

for (speaker, text), g, l in zip(held_out_turns, held_out_gold, lstm_labels):
    if l["category"] != g:
        print(f"LSTM miss: {text!r}  (gold {g}, got {l['category']})")

This task mostly needs the previous turn, not a long memory.
We don't claim the trained LSTM remembers further back than a plain RNN.

A decision stated as a plain fact is its weak spot -- even the keyword rule catches
"Venue is confirmed" when the model misses it. Neither is perfect. That's the honest result.

### 7b. Catch it out yourself (live mic)
**Predict:** will the model do better on a request, a proposal, or a plain fact?

Say one of each, one per cell:
- a request ("Can you send the invoice today?")
- a proposal, not yet agreed to ("How about we meet at the library?")
- a plain fact that's really a decision ("The venue is confirmed for Friday.")

In [ ]:
REQUEST_LIVE, _, _ = live_mic.say("block7_request", model, seconds=5)
print("Heard:", REQUEST_LIVE)

In [ ]:
PROPOSAL_LIVE, _, _ = live_mic.say("block7_proposal", model, seconds=5)
print("Heard:", PROPOSAL_LIVE)

In [ ]:
FACT_LIVE, _, _ = live_mic.say("block7_fact", model, seconds=5)
print("Heard:", FACT_LIVE)

from meeting_minutes import label_turns_keywords

INTENDED = {"request": "action_item", "proposal": "discussion", "plain fact": "decision"}

for kind, text in [("request", REQUEST_LIVE), ("proposal", PROPOSAL_LIVE), ("plain fact", FACT_LIVE)]:
    turns_live7 = [("You", text)]
    probs_live7 = label_probabilities(turns_live7, trained)[0]
    top_cat, top_p = max(zip(trained["labels"], probs_live7), key=lambda lp: lp[1])
    intended_cat = INTENDED[kind]
    intended_p = probs_live7[trained["labels"].index(intended_cat)]
    loss7 = -np.log(intended_p)   # loss IF this is the right reading -- large means caught out
    kw_cat = label_turns_keywords(turns_live7)[0]["category"]
    print(f"{kind:10s}: model -> {top_cat} ({top_p:.0%})   "
          f"loss if it should be {intended_cat!r} = {loss7:.2f}   keyword rule -> {kw_cat}")

**Why:** a request or proposal usually has a word to key on ("can you", "let's").
A plain fact has neither. That's the model's weak spot -- visible on what YOU said.

**The "Sure." test -- back to the hook.** "Can you book the room?" "Sure."
and "Shall we move it to Friday?" "Sure." Same two words, three ways:

In [ ]:
sure_cases = [
    [("Ravi", "Sure.")],
    [("Mei", "Can you book the room?"), ("Ravi", "Sure.")],
    [("Mei", "Shall we move it to Friday?"), ("Ravi", "Sure.")],
]
for case in sure_cases:
    case_text = chr(10).join(f"{s}: {t}" for s, t in case)
    case_turns, case_output, case_ends = run_pipeline(transcript=case_text)
    case_probs = label_probabilities(case_turns, trained)[-1]
    lead_in = case[0][1] if len(case) > 1 else "(no lead-in)"
    probs_str = ", ".join(f"{l} {p:.2f}" for l, p in sorted(zip(trained["labels"], case_probs), key=lambda x: -x[1]))
    print(f"{lead_in:28s} -> {probs_str}")

"Sure." alone could be anything. After the request, confidently an action item -- right.
After the proposal, confidently **discussion** -- **wrong**: it's really a decision.

The model reads context, same as the room, but doesn't always land correctly.

### The same word, two different answers (live mic) -- back to the hook
**Predict:** does the model read "Sure." differently depending on what came before?

Say these two exchanges, ~2s pause before each "Sure.":
- "Can you book the room?" / (pause) / "Sure."
- "Shall we move it to Friday?" / (pause) / "Sure." 

In [ ]:
SURE_REQ_LIVE, sure_req_path, _ = live_mic.say("sure_request", model, seconds=8)
print("Heard:", SURE_REQ_LIVE)

In [ ]:
SURE_PROP_LIVE, sure_prop_path, _ = live_mic.say("sure_proposal", model, seconds=8)
print("Heard:", SURE_PROP_LIVE)

def label_sure_exchange(path, pause_gap=1.2):
    result = model.transcribe(path, word_timestamps=True, language="en")
    words_ts = [{"start": w["start"], "end": w["end"], "text": w["word"]}
                for seg in result["segments"] for w in seg.get("words", [])]
    turn_text = segments_to_transcript(words_ts, pause_gap=pause_gap)
    lines = [l for l in turn_text.strip().split(chr(10)) if l]
    turns_sure = [(f"Speaker{i % 2 + 1}", l.split(": ", 1)[1]) for i, l in enumerate(lines)]
    probs_sure = label_probabilities(turns_sure, trained) if len(turns_sure) > 1 else None
    return turns_sure, probs_sure

for lead_in, path in [("after the request", sure_req_path), ("after the proposal", sure_prop_path)]:
    turns_sure, probs_sure = label_sure_exchange(path)
    if len(turns_sure) < 2:
        print(f"{lead_in:20s}: heard it as one turn. Leave a 2s pause, then try again.")
        continue
    found_sure = False
    for (speaker, text), p in zip(turns_sure, probs_sure):
        if text.strip().lower().rstrip(".") == "sure":
            top_cat, top_p = max(zip(trained["labels"], p), key=lambda lp: lp[1])
            print(f"{lead_in:20s}: 'Sure.' -> {top_cat} ({top_p:.0%})")
            found_sure = True
    if not found_sure:
        print(f"{lead_in:20s}: didn't hear a separate 'Sure.' turn. Leave a 2s pause, try again.")

**Why:** same word, read differently depending on what came before -- same as the hook.
On your own voice this may land differently; it's a genuine knife-edge case, not a sure thing.

### YOUR TURN (Part B): the LSTM notebook update
`new notebook = (how much to keep) x (old notebook) + (how much to add) x (new note)`

`f` eraser, `i` pencil, `g` new note, `c_prev` old notebook. Write that one line.

In [ ]:
def lstm_cell_update(f, i, g, c_prev):
    # TODO: return the new notebook (the line above, in numpy)
    return None

hands_on.lstm_cell_update = lstm_cell_update
hands_on.FORGET_KNOB = 0          # try -6, -3, 0, 3, 6: negative erases more, positive keeps more
hands_on.part_b()

<details><summary>Stuck? (click to expand)</summary>

- **Bigger numbers than expected** -- you used `+` where `*` should be.
- **Close, not quite** -- check `g` is multiplied in (`i * g`, not just `i`).
- **Still `None`** -- replace `return None` with your formula.

</details>

Turn the knob to **-6** and **+6**. What happens to the first word, and the score?
Why do both extremes hurt it?

**All three settings, compared.** "Keep a lot" (+6) is NOT better memory --
the model learned its own setting at 0, so either direction makes it worse.

<details><summary>More detail: why each extreme hurts, and the link to the retention graph</summary>

Knob -6, 0, +6 on the test meetings: 10/45, 35/45, 22/45. *(One ensemble member -- the Sure.
test above averages all five.)*

- **-6 hurts**: erases everything, including the line "Sure." needs. The toy gap dies at word 2.
- **+6 hurts**: barely erases anything. Old content piles up and swamps new words -- the gap
  actually **grows**, from 0.17 to 0.37.

Knob 0 sits at `sigmoid(0) = 0.5` (the 0.49 retention curve). Knob +3 sits at
`sigmoid(3) = 0.953` (the 0.95 curve). Same dial, same graph.

</details>

In [ ]:
for knob in (-6, 0, 6):
    hands_on.FORGET_KNOB = knob
    hands_on.part_b()
    print()
hands_on.FORGET_KNOB = 0   # leave the knob back at its trained setting

### 8b. Turn the knob on YOUR exchange (live mic)
**Predict:** will the knob flip the trained model's label on your own exchange too?

Say a short two-turn exchange, ~2s pause -- e.g. "Can you confirm the venue?" / "Sure." 

In [ ]:
EXCHANGE_LIVE, exchange_path, _ = live_mic.say("block8_exchange", model, seconds=8)
print("Heard (flat):", EXCHANGE_LIVE)

turns_live8, _ = label_sure_exchange(exchange_path)
print("Turns:", turns_live8)

if len(turns_live8) < 2:
    print("Heard it as one turn. Leave a 2s pause, then try this prompt again.")
else:
    def predict_with_knob(turns, knob):
        # Knob applied to all 5 members, then averaged -- same ensembling as
        # label_probabilities, so this agrees with the Sure. test above.
        tokens, ends = tokenize_transcript(turns)
        all_probs = []
        for member in trained["members"]:
            original_b_f = {n: getattr(member["model"], f"layer{n}").b_f.copy() for n in range(1, 6)}
            for n in range(1, 6):
                layer_n = getattr(member["model"], f"layer{n}")
                layer_n.b_f = original_b_f[n] + knob
            ids = [vocab.get(w, 0) for w in tokens]
            x = member["embeddings"][ids]
            out, _ = member["model"].forward(pad_to_multiple_of_4(x))
            scores = out[:len(x)][ends] @ member["head_W"].T + member["head_b"]
            e = np.exp(scores - scores.max(axis=1, keepdims=True))
            all_probs.append(e / e.sum(axis=1, keepdims=True))
            for n in range(1, 6):
                getattr(member["model"], f"layer{n}").b_f = original_b_f[n]   # restore right away
        return np.mean(all_probs, axis=0)

    for knob in (-6, 0, 6):
        probs8 = predict_with_knob(turns_live8, knob)
        labels8 = [max(zip(trained["labels"], p), key=lambda lp: lp[1]) for p in probs8]
        print(f"Knob {knob:+d}:", ", ".join(f"{s}: {lbl} ({p:.0%})" for (s, t), (lbl, p) in zip(turns_live8, labels8)))

**Why:** unlike the static sweep (one member), this cell averages all five --
the same ensembling used everywhere else. The knob can flip a label that was right at 0.

### 8c. Same test, through the toy LSTM's dial (recordings from 5c)
**See it:** same short/long conversations, toy LSTM, knob +3 -- untrained, not our trained model.

In [ ]:
def toy_lstm_kept(tokens_a, tokens_b, knob, threshold=50):
    # pct can exceed 100% if the knob is so high the notebook keeps adding
    # without erasing. The verdict is always just KEPT or NOT KEPT.
    layer = SimpleLSTM(8, 8, seed=5)
    layer.b_f = layer.b_f + knob
    ha = hb = ca = cb = np.zeros(8)
    gaps = []
    for ta, tb in zip(tokens_a, tokens_b):
        xa, xb = live_mic.word_vector(ta), live_mic.word_vector(tb)
        ha, ca = layer.step(xa, ha, ca)
        hb, cb = layer.step(xb, hb, cb)
        gaps.append(np.abs(ca - cb).sum())
    ref = gaps[toy_memory.line_boundary_index(tokens_a)]
    pct = gaps[-1] / ref * 100
    verdict = "KEPT" if pct >= threshold else "NOT KEPT"
    if pct > 150:
        verdict += " (it grew instead of fading)"
    return pct, verdict

short_pct_lstm, short_verdict_lstm = toy_lstm_kept(short_tokens, short_tokens_b, knob=3)
long_pct_lstm, long_verdict_lstm = toy_lstm_kept(long_tokens, long_tokens_b, knob=3)

plt.figure(figsize=(5, 3.5))
plt.bar(["short exchange", "long conversation"], [1 if short_pct_lstm >= 50 else 0, 1 if long_pct_lstm >= 50 else 0],
        color=["green" if short_pct_lstm >= 50 else "red", "green" if long_pct_lstm >= 50 else "red"])
plt.yticks([0, 1], ["NOT KEPT", "KEPT"])
plt.title("Knob +3: is line 1's fact still remembered?")
plt.tight_layout()
plt.show()
print(f"Short exchange, knob +3: {short_verdict_lstm}")
print(f"Long conversation, knob +3: {long_verdict_lstm}")

**Why:** both settings (keep-rate 0.9, knob +3) are stated, untrained numbers.
A long enough conversation outruns even a generous setting. Not a claim the LSTM beats an RNN --
same point either way: **plain memory degrades over a long conversation.**

### YOUR TURN: set the dial so the long conversation is remembered
**Predict:** is there a knob value where the long conversation flips to KEPT?

Pick a `KNOB` so the long conversation comes back **KEPT**. Edit and re-run.

In [ ]:
KNOB = 3   # YOUR TURN: try a bigger number if the check below says "not yet"

retained_pct, verdict = toy_lstm_kept(long_tokens, long_tokens_b, knob=KNOB)
print(f"Knob {KNOB:+d}: {verdict}")
if "KEPT" in verdict and "NOT" not in verdict:
    print("Check: OK. You found a setting where the first line survives.")
else:
    print("Not yet. Try a bigger KNOB (more positive keeps more).")

<details><summary>Stuck? (click to expand)</summary>

- Still "NOT KEPT" at 3? Try 5, then 8.
- Once it flips to "KEPT", bigger numbers tend to stay "KEPT" too.
- This cell doesn't affect anything below -- leave `KNOB` wherever you found it.

</details>

## 5. Meeting minutes
Your two-speaker transcript goes through the trained model.
Every turn gets a label: **decision**, **action item**, or **discussion**. Minutes get written from that.

In [ ]:
from IPython.display import Markdown, display
from lstm_model import run_pipeline
from meeting_minutes import label_turns_model, generate_minutes

turns, output, turn_ends = run_pipeline(transcript=transcript)
labels = label_turns_model(turns, output, turn_ends)
for (speaker, text), label in zip(turns, labels):
    print(f"{label['category']:<12} {speaker}: {text}")
print()
display(Markdown(generate_minutes(turns, labels)))

**Look closely.** Did it find both decisions? It misses decisions stated as plain facts,
like "Venue is confirmed" -- the model's weakest spot.

### Your own meeting
Write 5 lines, one turn per line: `Name: what they said`.
Labels optional -- if you add them, label every line `act`/`dec`/`dis`.

<details><summary>More detail: accuracy, vocabulary, and the default example</summary>

Tested on 17 held-out meetings: about 0.85-0.88 of turns right. Unusual words read as
`<unk>` and ignored -- names too, since the model never sees them.

The default below is the hook's own exchange, so there's a real answer when asked which
earlier line the model should have remembered.

</details>

In [ ]:
import os
import tempfile
from lstm_model import load_trained_model, label_probabilities
from pipeline import load_text, unknown_words
from IPython.display import Markdown, display

MY_MEETING = """
dis  Mei: Shall we move it to Friday?
dec  Ravi: Sure.
act  Mei: Can you book the room?
act  Ravi: Sure.
dis  Mei: Great, thanks.
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8") as f:
    f.write(MY_MEETING)
    my_meeting_path = f.name
try:
    my_text, my_gold = load_text(my_meeting_path)
except ValueError as e:
    # Strip the temp-file path -- a participant only needs to see what to fix.
    message = str(e).replace(my_meeting_path, "").strip(" ,:")
    print("Couldn't read that meeting:", message)
    print("Hint: label every line or none of them. Write each line as 'Name: text' -- something after the colon.")
    my_text = None
finally:
    os.remove(my_meeting_path)

if my_text is not None:
    my_turns, my_output, my_ends = run_pipeline(transcript=my_text)
    my_labels = label_turns_model(my_turns, my_output, my_ends)
    my_probs = label_probabilities(my_turns, trained)

    for i, ((speaker, text), label, probs) in enumerate(zip(my_turns, my_labels, my_probs)):
        model_cat = label["category"]
        confidence = probs[trained["labels"].index(model_cat)]
        if my_gold:
            your_cat = my_gold[i]
            mark = "" if your_cat == model_cat else "  <- X"
            print(f"{your_cat:<12} (you)   {model_cat:<12} (model, {confidence:.0%} sure)   {speaker}: {text}{mark}")
        else:
            print(f"{model_cat:<12} (model, {confidence:.0%} sure)   {speaker}: {text}")
    if not my_gold:
        print()
        print("No labels given. Add act/dec/dis at the start of each line and run again.")

    print()
    display(Markdown(generate_minutes(my_turns, my_labels)))

    unknown = unknown_words(my_turns)
    print()
    if unknown:
        print(f"Words the model never saw in training: {', '.join(unknown)}")
    vocab_size = len(load_trained_model()["vocab"])
    print(f"The model knows {vocab_size} words. Anything else is <unk>.")

    print()
    print("Pick one line it got wrong. Which EARLIER line should it have remembered?")
    print("(Default example: the first \"Sure.\" is near 50-50. It needs \"Shall we move it")
    print("to Friday?\" read as agreeing to a plan.)")

**Add a line before it, and watch it change.** Put one more line before the whole meeting:

In [ ]:
if my_text is not None:
    with_context = "Mei: Let's plan the showcase." + chr(10) + my_text
    with_context_turns, with_context_output, with_context_ends = run_pipeline(transcript=with_context)
    with_context_probs = label_probabilities(with_context_turns, trained)

    sure_count = 0
    sure_labels = ["first \"Sure.\" (after the proposal)", "second \"Sure.\" (after the request)"]
    for (speaker, text), probs in zip(with_context_turns, with_context_probs):
        if text == "Sure.":
            top = max(zip(trained["labels"], probs), key=lambda lp: lp[1])
            which = sure_labels[sure_count] if sure_count < len(sure_labels) else f"Sure. #{sure_count + 1}"
            print(f"{speaker}: {which:32s} -> {top[0]} ({top[1]:.0%})")
            sure_count += 1

One extra line earlier flips the model's answer on that first "Sure." --
from near-50-50 to a correct `decision`, 56% against 43%. One example, not a general rule --
an earlier line can just as easily push the wrong way.

### 9b. Talk your own meeting (live mic)
**Predict:** will spoken minutes look like the ones you'd write by hand?

Talk through 4-6 lines, alone or with a partner, ~2s pause between lines --
a decision, an action item, some plain discussion. Up to 35s.

In [ ]:
MEETING_LIVE, meeting_path, _ = live_mic.say("block9_meeting", model, seconds=35, minimum_words=6)
print("Heard (flat):", MEETING_LIVE)

meeting_result = model.transcribe(meeting_path, word_timestamps=True, language="en")
meeting_words_ts = [{"start": w["start"], "end": w["end"], "text": w["word"]}
                     for seg in meeting_result["segments"] for w in seg.get("words", [])]
live_meeting_text = segments_to_transcript(meeting_words_ts, pause_gap=1.2)
print(live_meeting_text)

### YOUR TURN: label your meeting before the model does
One label per turn, same order as above: `act`, `dec`, or `dis`.

In [ ]:
_live_meeting_turn_count = len([l for l in live_meeting_text.strip().split(chr(10)) if l])
MY_LABELS = ["dis"] * _live_meeting_turn_count   # YOUR TURN: your own act/dec/dis guesses
_MY_LABELS_DEFAULT = ["dis"] * _live_meeting_turn_count

if len(MY_LABELS) != _live_meeting_turn_count:
    print(f"Not yet. {len(MY_LABELS)} labels for {_live_meeting_turn_count} turns. One label per turn.")
    MY_LABELS_OK = False
elif any(l not in ("act", "dec", "dis") for l in MY_LABELS):
    print("Not yet. Each label must be 'act', 'dec' or 'dis'.")
    MY_LABELS_OK = False
elif MY_LABELS == _MY_LABELS_DEFAULT:
    print("Add your labels first -- change MY_LABELS above, then run this cell again.")
    MY_LABELS_OK = False
else:
    print("Check: OK. Compared against the model's own labels below.")
    MY_LABELS_OK = True

<details><summary>Stuck? (click to expand)</summary>

- Wrong count -- count the turns above, match exactly.
- Typo -- only `act`, `dec` or `dis`, lowercase.
- Skipping this is fine. The model's labels still print.

</details>

In [ ]:
live_meeting_turns, live_meeting_output, live_meeting_ends = run_pipeline(transcript=live_meeting_text)
live_meeting_labels = label_turns_model(live_meeting_turns, live_meeting_output, live_meeting_ends)
LABEL_MAP = {"act": "action_item", "dec": "decision", "dis": "discussion"}
for i, ((speaker, text), label) in enumerate(zip(live_meeting_turns, live_meeting_labels)):
    model_cat = label["category"]
    if MY_LABELS_OK:
        your_cat = LABEL_MAP[MY_LABELS[i]]
        mark = "" if your_cat == model_cat else "  <- different"
        print(f"{your_cat:<12} (you)   {model_cat:<12} (model)   {speaker}: {text}{mark}")
    else:
        print(f"{model_cat:<12} {speaker}: {text}")
print()
display(Markdown(generate_minutes(live_meeting_turns, live_meeting_labels)))

unknown9 = unknown_words(live_meeting_turns)
if unknown9:
    print()
    print(f"Words the model never saw in training: {', '.join(unknown9)}")

**Why:** same pipeline as the typed cell above, just from your own voice.
No gold labels to compare against here -- just what the model made of it.
A mistimed pause can still split or merge turns wrong.

**See it:** add a "Can you book the room? / Sure." ending.
Does the last answer move more when you blank line 1, or the line right before it?

In [ ]:
if len(live_meeting_turns) >= 6:
    reach_turns = live_meeting_turns
else:
    # A short meeting (under 6 turns) doesn't show this reliably.
    print(f"Your meeting only had {len(live_meeting_turns)} turns. Using the longer sample meeting instead.")
    reach_turns = turns   # the block 1-2 sample meeting, always 9 turns

last_speaker, last_text = reach_turns[-1]
if len(last_text.split()) <= 4 and len(reach_turns) >= 2:
    # Already ends with a short, context-dependent reply -- use it as-is.
    test_turns = list(reach_turns)
else:
    test_turns = list(reach_turns) + [("Mei", "Can you book the room?"), ("Ravi", "Sure.")]

def _blank(text):
    # Words to <unk>, same as the README's method -- removes the line's own
    # content without changing the turn count or the <eot> markers.
    return " ".join(["<unk>"] * len(text.split()))

def reach_test(all_turns, replace_idx):
    modified = list(all_turns)
    speaker, text = modified[replace_idx]
    modified[replace_idx] = (speaker, _blank(text))
    return label_probabilities(modified, trained)[-1]

baseline_last = label_probabilities(test_turns, trained)[-1]
changed_first = reach_test(test_turns, 0)
changed_before_last = reach_test(test_turns, len(test_turns) - 2)

shift_first = float(np.abs(changed_first - baseline_last).sum() / 2)      # total variation distance, 0-1
shift_before_last = float(np.abs(changed_before_last - baseline_last).sum() / 2)

plt.figure(figsize=(5, 3.5))
bars = plt.bar(["change line 1", "line before last"],
               [shift_first * 100, shift_before_last * 100], color=["steelblue", "darkorange"])
plt.ylabel("how much the last answer moved")
plt.title("The trained model's real memory reach")
plt.ylim(0, 100)
for bar, v in zip(bars, [shift_first * 100, shift_before_last * 100]):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 2, f"{v:.0f}%", ha="center")
plt.tight_layout()
plt.show()
print(f"Changing line 1: last answer moved {shift_first:.0%}")
print(f"Changing the line before last: last answer moved {shift_before_last:.0%}")

**Why:** this is the real trained ensemble, not a toy. Even the trained model really
only uses about the previous turn -- blanking line 1 barely moves the last answer, blanking
the line right before it moves it a lot. We don't claim it beats a plain RNN at this scale.

## 6. Wrap-up
**In one sentence:** sound becomes words, memory turns words into meaning, meaning becomes minutes.

**Three things it still gets wrong:**
1. A decision stated as fact ("Venue is confirmed...") with no "agree" to catch.
2. Words it never saw -- read as `<unk>`.
3. A bare "Sure." after a proposal -- reads context, doesn't always land right.

Take-home: write 5 lines of your own (section 5). The training code is with the facilitators.

### Going further
- Try `small.en` or `medium.en` in section 1. How much better, how much slower?
- Record a real two-person conversation and run it through `get_transcript`.
- Section 2 again, with your own noise or recording.

### YOUR TURN: explain it in your own words *(bonus)*
In 1-2 sentences: why does a long conversation's memory fade more than a short one's?

In [ ]:
MY_WHY = """YOUR TURN: write your own 1-2 sentence explanation here."""
print(MY_WHY)

### What I built today

In [ ]:
print("What I built today:")

_DEFAULTS = {
    "KNOB": 3,
    "MY_SENTENCE_A": "Cats are sitting by the door tonight.",
    "MY_SENTENCE_B": "Dogs are sitting by the door tonight.",
    "MY_WHY": "YOUR TURN: write your own 1-2 sentence explanation here.",
}

_recorded_vars = ["SAID_LIVE", "NOISY_LIVE", "WORD_LIVE", "SENTENCE_LIVE", "CATS_LIVE", "DOGS_LIVE",
                   "SHORT_A", "SHORT_B", "LONG_A", "LONG_B", "GATE_LIVE", "REQUEST_LIVE", "PROPOSAL_LIVE",
                   "FACT_LIVE", "SURE_REQ_LIVE", "SURE_PROP_LIVE", "EXCHANGE_LIVE", "MEETING_LIVE",
                   "SHORT_CONV", "LONG_CONV"]
_recorded_count = sum(1 for v in _recorded_vars if v in globals())
print(f"- {_recorded_count}/{len(_recorded_vars)} live-mic prompts recorded (or used the fallback for)")

if "KNOB" in globals():
    if KNOB == _DEFAULTS["KNOB"]:
        print("- the memory dial: (skipped)")
    else:
        print(f"- your memory dial: KNOB = {KNOB!r}")

if "MY_SENTENCE_A" in globals() and "MY_SENTENCE_B" in globals():
    if MY_SENTENCE_A == _DEFAULTS["MY_SENTENCE_A"] and MY_SENTENCE_B == _DEFAULTS["MY_SENTENCE_B"]:
        print("- your first-word pair: (skipped)")
    else:
        print(f"- your first-word pair: {MY_SENTENCE_A!r} / {MY_SENTENCE_B!r}")

if "MY_LABELS" in globals():
    if globals().get("MY_LABELS_OK"):
        print(f"- your meeting's labels: {MY_LABELS!r}")
    else:
        print("- your meeting's labels: (skipped)")

if "MY_WHY" in globals():
    if MY_WHY.strip() == _DEFAULTS["MY_WHY"]:
        print("- your explanation: (skipped)")
    else:
        print(f"- your explanation: {MY_WHY.strip()[:120]!r}")